In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
os.environ["CORRFUNC_USE_AVX512"] = "0"
os.environ["CORRFUNC_USE_AVX2"] = "0"
os.environ["CORRFUNC_USE_AVX"] = "0"
from Corrfunc.mocks.DDtheta_mocks import DDtheta_mocks
import astropy.units as u
import healpy as hp
from joblib import Parallel, delayed
import argparse
import configparser
from scipy.optimize import curve_fit
from scipy.signal import find_peaks, savgol_filter
from scipy import integrate
from scipy.interpolate import UnivariateSpline
import scipy.stats as stats
import astropy.constants as const

In [2]:
def parse_config():    
    parser = argparse.ArgumentParser()    
    parser.add_argument('--config', required=True)    
    args = parser.parse_args(["--config", "config.ini"])
    config = configparser.ConfigParser() 
    config.read(args.config) 
    return config
    
def make_clean():    
    global galaxy_csv_name, GW_csv_name, GW_data_size, galaxy_data_size, redshift_min, redshift_bin_size, luminosity_bin_min, luminosity_bin_size, theta_max, nside, z_bin_num, d_L_bin_num, r2dr, nthreads
    galaxy_csv_name = config['Config'].get('galaxy_csv_name')
    GW_csv_name = config['Config'].get('GW_csv_name')
    GW_data_size = config['Config'].getint('GW_data_size')    
    galaxy_data_size = config['Config'].getint('galaxy_data_size')    
    redshift_min = config['Config'].getfloat('redshift_min')    
    redshift_bin_size = config['Config'].getfloat('redshift_bin_size')
    luminosity_bin_min = config['Config'].getfloat('luminosity_bin_min')
    luminosity_bin_size = config['Config'].getfloat('luminosity_bin_size')
    theta_max = config['Config'].getfloat('theta_max')
    nside = config['Config'].getint('nside')
    z_bin_num = config['Config'].getint('z_bin_num')
    d_L_bin_num = config['Config'].getint('d_L_bin_num')
    r2dr = config['Config'].getint('random_2_data_ratio')
    nthreads = config['Config'].getint('nthreads')

In [3]:
#get variables from config file
config = parse_config()
make_clean()

In [4]:
#csv data
galaxy_df_full = pd.read_csv(galaxy_csv_name)
GW_df_full = pd.read_csv(GW_csv_name)

In [5]:
#testing parameters
if GW_data_size > len(GW_df_full):
    raise ValueError("GW_data_size too big. Input must be <=", len(GW_df_full))
if int(GW_data_size) != GW_data_size or GW_data_size < 1:
    raise ValueError("GW_data_size must be a positive integer")
if galaxy_data_size > len(galaxy_df_full):
    raise ValueError("galaxy_data_size too big. Input must be <=", len(galaxy_df_full))
if int(galaxy_data_size) != galaxy_data_size or galaxy_data_size < 1:
    raise ValueError("galaxy_data_size must be a positive integer")
if redshift_min < 0:
    raise ValueError("redshift_min must be positive")
if redshift_bin_size < 0:
    raise ValueError("redshift_bin_size must be positive")
if luminosity_bin_min < 0:
    raise ValueError("luminosity_bin_min must be positive")
if luminosity_bin_size < 0:
    raise ValueError("luminosity_bin_size must be positive")
if theta_max < 0:
    raise ValueError("theta_max must be positive")
if nside <= 0 or (nside & (nside - 1)) != 0:
    raise ValueError("nside must be a power of 2")
if int(z_bin_num) != z_bin_num or z_bin_num < 1:
    raise ValueError("z_bin_num must be a positive integer")
if int(d_L_bin_num) != d_L_bin_num or d_L_bin_num < 1:
    raise ValueError("d_L_bin_num must be a positive integer")
if r2dr < 1:
    raise ValueError("r2dr must be larger than 1")

In [6]:
#pdf for GW events from GWTC-4
gamma = stats.gamma
GW_a, GW_loc, GW_scale = 1.9094, 0, 1319.36
galaxy_a, galaxy_loc, galaxy_scale = 2.4211, 0, 0.1753
GW_probs = gamma.pdf(GW_df_full['luminosity distance'], GW_a, GW_loc, GW_scale)
galaxy_probs = gamma.pdf(galaxy_df_full['redshift'], galaxy_a, galaxy_loc, galaxy_scale)

In [7]:
#print(GW_probs)
#print(np.sum(GW_probs))
GW_probs = GW_probs / np.sum(GW_probs)
galaxy_probs = galaxy_probs / np.sum(galaxy_probs)
print(np.sum(GW_probs))
print(np.sum(galaxy_probs))

0.9999999999999998
1.0


In [8]:
#random sample of GW and galaxy data sets
z_least_bin_count = 0
while z_least_bin_count < 2:
    GW_df_indices = np.random.choice(len(GW_df_full), GW_data_size, replace=False, p=GW_probs)
    GW_df = GW_df_full.iloc[GW_df_indices]
    galaxy_df_indices = np.random.choice(len(galaxy_df_full), galaxy_data_size, replace=False, p=galaxy_probs)
    galaxy_df = galaxy_df_full.iloc[galaxy_df_indices]
    z_least_bin_count = (galaxy_df['redshift'] < redshift_bin_size).sum()

In [9]:
#defines the redshift bin for an iteration
def redshift_bin(i):
    z_min = redshift_min + i * redshift_bin_size
    z_max = z_min + redshift_bin_size
    z_bin = galaxy_df[galaxy_df['redshift'] > z_min]
    z_bin = z_bin[z_bin['redshift'] < z_max]
    return z_bin

In [10]:
#defines the luminosity distance bin for an iteration
def luminosity_distance_bin(i):
    d_L_min = luminosity_bin_min + i * luminosity_bin_size
    d_L_max = d_L_min + luminosity_bin_size
    filtered_GW = GW_df[GW_df['luminosity distance'] < d_L_max]
    filtered_GW = filtered_GW[filtered_GW['luminosity distance'] > d_L_min]
    d_L_bin = filtered_GW
    return d_L_bin

In [11]:
#creates an array of ra and dec values the length of num
def random_func(num):
    ra_random = np.random.uniform(0,360,num)
    u = np.random.uniform(-1,1,num)
    dec_random = np.degrees(np.arcsin(u))
    random_coords = np.array([np.array(ra_random),np.array(dec_random)])
    return random_coords

In [12]:
#creates bins using the random dataset that is proportional to the size of the "real" dataset
def random_bins(random_coords, bin_size):
    random_indices = np.random.choice(len(random_coords[0]), bin_size, replace=False)

    random_ra = random_coords[0][random_indices]
    random_dec = random_coords[1][random_indices]
    random_coords_filtered = np.array([random_ra, random_dec])
    return random_coords_filtered

In [13]:
#cross correlation function
def dd_rr_func(theta_bin, galaxy, GW):
    galaxy_ra, galaxy_dec = galaxy
    GW_ra, GW_dec = GW
    dd_rr = DDtheta_mocks(
        autocorr=0,
        nthreads=nthreads, 
        binfile=theta_bin,
        RA1=galaxy_ra,
        DEC1=galaxy_dec,
        RA2=GW_ra,
        DEC2=GW_dec
    )
    return dd_rr

In [14]:
#normalizes dd and rr based on the bin sizes
def norm(dd, rr, z_bin, d_L_bin):
    dd_norm = dd['npairs'] / (len(z_bin) * len(d_L_bin))
    rr_norm = rr['npairs'] / (r2dr * len(z_bin) * r2dr * len(d_L_bin))
    return dd_norm, rr_norm

#Peebles-Hauser estimator
def w_func(dd_norm, rr_norm):
    w = (dd_norm / rr_norm) - 1
    return w

In [15]:
#w calculation without jackknife estimate
def w_no_jk(z_bin, d_L_bin, all_data):
    galaxy, GW, galaxy_random, GW_random = all_data
    dd = dd_rr_func(theta_bin, galaxy, GW)
    rr = dd_rr_func(theta_bin, galaxy_random, GW_random)
    dd_norm, rr_norm = norm(dd, rr, z_bin, d_L_bin)
    w = w_func(dd_norm, rr_norm)
    return w

In [16]:
#assigns healpix regions
def assign_healpix_region(ra_deg, dec_deg):
    theta = np.radians(90.0 - dec_deg)  # co-latitude
    phi = np.radians(ra_deg)            # longitude
    return hp.ang2pix(nside, theta, phi)

In [17]:
#assigns region numbers to each data point based on the nside resolution
def regions(coords):
    ra, dec = coords
    assigned_regions = assign_healpix_region(ra, dec)
    return assigned_regions

In [18]:
#masks the data by removing one region
def masking(coords, region, i):
    ra, dec = coords

    mask = region != i

    sample_ra = ra[mask]
    sample_dec = dec[mask]
    sample = np.array([sample_ra, sample_dec])
    return sample

In [19]:
#performs the w calculation after removing one region at a time, returns the w value without jackknife if no points are found in the region
def jk_loop(all_data, all_regions, i, w0):
    galaxy, GW, galaxy_random, GW_random = all_data
    galaxy_regions, GW_regions, galaxy_random_regions, GW_random_regions = all_regions

    if i in galaxy_regions or i in GW_regions:
        galaxy_sample = masking(galaxy, galaxy_regions, i)
        GW_sample = masking(GW, GW_regions, i)
        galaxy_random_sample = masking(galaxy_random, galaxy_random_regions, i)
        GW_random_sample = masking(GW_random, GW_random_regions, i)

        galaxy_random_ra, galaxy_random_dec = galaxy_random_sample
    
        dd = dd_rr_func(theta_bin, galaxy_sample, GW_sample)
        rr = dd_rr_func(theta_bin, galaxy_random_sample, GW_random_sample)
    
        dd_norm, rr_norm = norm(dd, rr, galaxy_sample[0], GW_sample[0])
        w = w_func(dd_norm, rr_norm)
    else:
        w = w0
    return w

In [20]:
#runs the jk_loop function and calculates the mean and variance from the results
def jackknife(all_data, all_regions, w0):
    galaxy, GW, galaxy_random, GW_random = all_data
    galaxy_regions, GW_regions, galaxy_random_regions, GW_random_regions = all_regions

    n_regions = len(galaxy_regions)

    jackknife_ws = Parallel(n_jobs=1)(
        delayed(jk_loop)(all_data, all_regions, i, w0) for i in range((nside**2)*12)
    )
    
    jackknife_mean = np.nanmean(jackknife_ws)
    jackknife_variance = ((n_regions - 1) / n_regions) * np.sum((jackknife_ws - jackknife_mean) ** 2)
    return jackknife_variance

In [21]:
#returns the w value and jackknife variance for pair of a redshift bin and a luminosity distance bin
def run_code(z_iter, d_L_iter):
    z_bin = redshift_bin(z_iter)
    d_L_bin = luminosity_distance_bin(d_L_iter)
    #print(len(z_bin))
    #print(len(d_L_bin))
    
    galaxy_ra, galaxy_dec = np.array(z_bin["RA_deg"]), np.array(z_bin["Dec_deg"])
    galaxy = np.array([galaxy_ra, galaxy_dec])
    GW_ra, GW_dec = np.array(d_L_bin["RA_deg"]), np.array(d_L_bin["Dec_deg"])
    GW = np.array([GW_ra, GW_dec])

    galaxy_random = random_bins(all_galaxy_random, len(z_bin)*r2dr)
    GW_random = random_bins(all_GW_random, len(d_L_bin)*r2dr)

    inputs = [galaxy, GW, galaxy_random, GW_random]
    results = Parallel(n_jobs=4)(delayed(regions)(data) for data in inputs)
    galaxy_regions, GW_regions, galaxy_random_regions, GW_random_regions = results

    all_data = [galaxy, GW, galaxy_random, GW_random]
    all_regions = [galaxy_regions, GW_regions, galaxy_random_regions, GW_random_regions]
    
    w = w_no_jk(z_bin, d_L_bin, all_data)
    var = jackknife(all_data, all_regions, w)
    return w, var

In [22]:
print(len(GW_df))
print(len(galaxy_df))

5000
100000


In [23]:
%%time
ws = []
variances = []

#establishes the theta bin for the cross correlation function
theta_max_deg = np.degrees(theta_max)
theta_bin = np.array([0, theta_max_deg]) #changed theta_min

galaxy_size = len(galaxy_df)
GW_size = len(GW_df)

#establishes the random datasets
galaxy_ra_random, galaxy_dec_random = random_func(galaxy_size*r2dr)
GW_ra_random, GW_dec_random = random_func(GW_size*r2dr)
all_galaxy_random = np.array([galaxy_ra_random, galaxy_dec_random])
all_GW_random = np.array([GW_ra_random, GW_dec_random])

#loops the run_code function over all pairs of redshift bins and luminosity distance bins
for d_L_iter in range(d_L_bin_num):
    for z_iter in range(z_bin_num):
        w, var = run_code(z_iter, d_L_iter)
        ws.append(w)
        variances.append(var)
        print(z_iter + 1, "/", z_bin_num)
    print(d_L_iter + 1, "/", d_L_bin_num)

print(ws)
print(len(ws))
print(variances)
print(len(variances))

[Warning] The CPU supports AVX2 but the compiler does not.  Can you try another compiler?
[Warning] The CPU supports AVX but the compiler does not.  Can you try another compiler?
[Warning] The CPU supports SSE4.2 but the compiler does not.  Can you try another compiler?
[Warning] The CPU supports SSE4.1 but the compiler does not.  Can you try another compiler?
[Warning] The CPU supports SSSE3 but the compiler does not.  Can you try another compiler?


1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
1 / 6
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
2 / 6
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
3 / 6
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
4 / 6
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
5 / 6
1 / 20
2 / 20
3 / 20
4 / 20
5 / 20
6 / 20
7 / 20
8 / 20
9 / 20
10 / 20
11 / 20
12 / 20
13 / 20
14 / 20
15 / 20
16 / 20
17 / 20
18 / 20
19 / 20
20 / 20
6 / 6
[array([-1.]), array([-1.]), array([1.12765957]), array([-

In [24]:
stdevs = np.sqrt(variances)

In [25]:
ws_arr = np.concatenate(ws)
stdevs_arr = np.array(stdevs)

In [26]:
print(ws_arr)
print(stdevs_arr)

[-1.00000000e+00 -1.00000000e+00  1.12765957e+00 -4.70899471e-01
 -4.42896936e-01  1.47540984e-01  4.04624277e-02 -2.24806202e-01
  1.31861913e-01  2.60338716e-01  5.26315789e-02 -4.90490490e-02
  4.25713069e-04 -4.69589326e-02  1.19881675e-02  3.66093688e-02
  2.30119284e-01  1.30695045e-01 -2.34720621e-01  3.68115666e-02
 -1.00000000e+00 -1.00000000e+00 -1.00000000e+00 -3.58974359e-01
 -2.17986315e-01  4.08450704e-01  3.95157981e-01  4.19389395e-01
  1.73249902e-01  4.13694722e-02  5.71785836e-02 -1.61911555e-01
  1.52608338e-01 -5.08684864e-02  3.45393837e-02  7.63807286e-02
 -1.04620750e-02  5.20903374e-02  1.67028679e-01 -4.05081253e-02
 -1.00000000e+00 -1.00000000e+00 -4.80519481e-01 -2.24828935e-02
  1.72227232e-01 -4.55991516e-02 -8.35579515e-02  8.61780855e-02
  2.92911280e-01  1.95873579e-01  1.64065355e-01  2.73134937e-02
  1.91907830e-02  7.23170627e-02  1.55618432e-02  7.01663202e-03
  5.96051407e-03  4.13632391e-02  5.11952292e-02 -2.75351340e-02
 -1.00000000e+00 -1.00000

In [27]:
output = np.column_stack((ws_arr, stdevs_arr))

from io import StringIO
buffer = StringIO()
config.write(buffer)
config_string = buffer.getvalue()

header_text = f"{config_string}\nws,stdevs"

np.savetxt("output_5.csv", output, delimiter=",", header=header_text, comments="")

In [28]:
from io import StringIO
buffer = StringIO()
config.write(buffer)
config_string = buffer.getvalue()

header_text = f"{config_string}\nfeature,score"

print(config["Config"])

<Section: Config>
